# Obesity Risk Prediction - Case 9

## 1. Introduction
This notebook implements a machine learning model to predict obesity risk based on the Kaggle Playground Series S4E2 dataset. 
It follows the specific requirements for **Case 9**, which mandates the use of **Gradient Boosting Classifier** with hyperparameter tuning via **GridSearchCV**.

### Dataset
The dataset consists of estimation of obesity levels based on eating habits and physical condition.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set plot style
sns.set(style="whitegrid")

## 2. Data Loading

In [ ]:
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
train_df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Target Variable Distribution
plt.figure(figsize=(10, 6))
sns.countplot(y='NObeyesdad', data=train_df, order=train_df['NObeyesdad'].value_counts().index)
plt.title('Distribution of Obesity Levels')
plt.show()

In [ ]:
# Numerical Features Distribution
numerical_cols = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
train_df[numerical_cols].hist(bins=15, figsize=(15, 10))
plt.suptitle('Distribution of Numerical Features')
plt.show()

## 4. Data Preprocessing

In [ ]:
X = train_df.drop(['id', 'NObeyesdad'], axis=1)
y = train_df['NObeyesdad']
X_test_raw = test_df.drop(['id'], axis=1)

# Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Define Preprocessing Pipeline
numerical_cols = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
categorical_cols = ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Split Data
X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Apply Preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test_raw)

print("Preprocessing complete.")

## 5. Modeling (Case 9)
**Algorithm**: Gradient Boosting Classifier
**Hyperparameters**:
- `n_estimators`: [50, 100, 200]
- `learning_rate`: [0.01, 0.1, 0.2]
- `max_depth`: [3, 5, 7]

In [ ]:
gbc = GradientBoostingClassifier(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

grid_search = GridSearchCV(estimator=gbc, param_grid=param_grid, cv=3, scoring='accuracy', verbose=2, n_jobs=-1)
grid_search.fit(X_train_processed, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_

print(f"Best Parameters: {best_params}")

## 6. Evaluation

In [ ]:
y_pred = best_model.predict(X_val_processed)
acc = accuracy_score(y_val, y_pred)

print(f"Validation Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=le.classes_))

## 7. Submission

In [ ]:
predictions = best_model.predict(X_test_processed)
predictions_decoded = le.inverse_transform(predictions)

submission = pd.DataFrame({'id': test_df['id'], 'NObeyesdad': predictions_decoded})
submission.to_csv('../submission.csv', index=False)
print("Submission saved to submission.csv")